# Get the libraries installed and imported

In [ ]:
!git clone https://github.com/davidhallac/TICC.git

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from sklearn.preprocessing import StandardScaler

In [ ]:
# Change the working directory to 'TICC'
os.chdir('TICC')

# Verify the change
print(os.getcwd())

In [ ]:
from TICC.TICC_solver import TICC

# Get the data ready

The economic indicators used are the following:


1. Balance of Payment (BOP). In the FRED database, I'm using the `BOPGTB` indicator, so only the Balance of Payments for goods is included. The unit is in millions of dollars.
2. Consumer Price Index (CPI) for All Urban Consumers, Food and Energy excluded. This is widely known as the core CPI. The code in the FRED database is `CPILFESL`. This index is normalized at 1982-1984 = 100.
3. Industrial Production (FRED database code: `INDPRO`). This is an index, normalized at 2017 = 100.
4. Money (M2) Supply (FRED database code: `M2SL`). The unit is in billions of dollars.
5. Producer Price Index by Commodity (FRED database code: `PPIACO`). The index is normalized at 1982 = 100.
6. Unemployment Rate (FRED database code: `UNRATE`). The index is reported as a percentage of the labor force, so the unit is in percent. This indicator is also known as the U-3 measure of labor underutilization.
7. Real interest rate (FRED database code: `REAINTRATREARAT10Y`). The unit is in percent.

Except for the PPI and Real interest rate, the other indicators are seasonally adjusted.


In [ ]:
bop = pd.read_csv('/content/BOPGTB.csv')
bop.head()

In [ ]:
df = bop.copy()
df = pd.concat([df,
                pd.read_csv('/content/CPILFESL.csv', usecols=[1]),
                pd.read_csv('/content/INDPRO.csv', usecols=[1]),
                pd.read_csv('/content/M2SL.csv', usecols=[1]),
                pd.read_csv('/content/PPIACO.csv', usecols=[1]),
                pd.read_csv('/content/UNRATE.csv', usecols=[1]),
                pd.read_csv('/content/REAINTRATREARAT10Y.csv', usecols=[1])], axis=1)
df.head()

In [ ]:
_, ax = plt.subplots(ncols=2, nrows=int(np.ceil(df.shape[1]/2)), figsize=(15,12))
for i, a in enumerate(ax.flat):
  try:
    sns.lineplot(data=df.iloc[:,(i+1)], ax=a)
    a.set_title(df.columns[i+1])
  except:
    pass
plt.tight_layout()
plt.savefig('mts.png')
plt.close('all')

In [ ]:
df.dropna(inplace=True)
df.tail()

In [ ]:
dates = df.pop('observation_date')
df_norm = StandardScaler().fit_transform(df)
np.savetxt('USecon_mts.csv', df_norm, delimiter=',')

In [ ]:
np.loadtxt('/content/TICC/USecon_mts.csv', delimiter=',').shape

# TICC

In [ ]:
WINDOW_SIZE = 2 # w
N_CLUSTERS = 10 # K
N_INDICATORS = 7 # n

In [ ]:
fname = "/content/TICC/USecon_mts.csv"
ticc = TICC(window_size=WINDOW_SIZE, number_of_clusters=N_CLUSTERS,
            lambda_parameter=11e-2, beta=10, maxIters=1000,
            compute_BIC=True, threshold=2e-5,
            write_out_file=False, prefix_string="content/", num_proc=1)
cluster_assignment, cluster_MRFs, bic = ticc.fit(input_file=fname)

# print(cluster_assignment)
# np.savetxt('Results.txt', cluster_assignment, fmt='%d', delimiter=',')

# Results

## Clusters or the economy states in the US

In [ ]:
# new cluster_bounds, which mark the beginning of a period
current_label = cluster_assignment[0]
cluster_bounds = [0]
for i in range(len(cluster_assignment)):
  if cluster_assignment[i] != current_label:
    cluster_bounds.append(i)
    current_label = cluster_assignment[i]
dates.iloc[cluster_bounds]

In [ ]:
_, ax = plt.subplots(figsize=(20,10))
df_norm = pd.DataFrame(df_norm, columns=['BOP', 'Core_CPI', 'Industrial_production',
                                         'M2', 'PPI', 'Unemployment_rate',
                                         'Real_interest_rate'])
sns.lineplot(df_norm, ax=ax)

for bound in cluster_bounds:
  plt.axvline(x=bound, linewidth=1, color='grey')
  plt.text(x=bound+2, y=4.0, s=int(cluster_assignment[bound])) # annotate the cluster label


plt.xticks(ticks=np.arange(0, len(df_norm), 12),
           labels=pd.to_datetime(dates).dt.year.unique())
plt.xticks(ticks=cluster_bounds,
           labels=dates.iloc[cluster_bounds], rotation=75, minor=True)
ax.tick_params(axis='x', which='minor', pad=8.0)
plt.ylabel('Normalized values')
plt.title('The economy of the US\nin the first quarter of the 21st century')
plt.legend(bbox_to_anchor=(1.1, 1), loc='upper right')

plt.savefig('mts_clustered.png')
plt.close('all')

## Markov Random Field (MRF)

In [ ]:
# depends on the N_CLUSTERS
cluster_MRFs.keys()

In [ ]:
N_CLUSTERS, WINDOW_SIZE

In [ ]:
# each MRF has the same shape
for key in cluster_MRFs.keys():
  print(key)
  print(cluster_MRFs[key].shape)

In [ ]:
for cluster in range(N_CLUSTERS):
  _, ax = plt.subplots(figsize=(12,10))
  sns.heatmap(cluster_MRFs[cluster], cmap='YlOrBr', annot=True, fmt='.2f', ax=ax)
  for i in range(0,(N_INDICATORS*WINDOW_SIZE), N_INDICATORS):
    ax.axvline(x=i, linewidth=1, color='grey')
    ax.axhline(y=i, linewidth=1, color='grey')
  ax.set_title(f'MRF {cluster}')
  plt.savefig(f'plot_MRF{cluster}.png');

## BIC

In [ ]:
bic

In [ ]:
fname = "/content/TICC/USecon_mts.csv"
bics = np.zeros((4, 4))
k_values = np.arange(2,6)
w_values = np.arange(3,7)
for k in range(4):
  for w in range(4):
    ticc = TICC(window_size=w_values[w], number_of_clusters=k_values[k],
                lambda_parameter=11e-2, beta=40, maxIters=1000,
                compute_BIC=True, threshold=2e-5,
                write_out_file=False, prefix_string="content/", num_proc=1)
    _, _, bic = ticc.fit(input_file=fname)
    bics[k, w] = bic

_, ax = plt.subplots(figsize=(14,12))
sns.heatmap(bics, cmap='YlOrBr', annot=True, fmt='.2f',
            xticklabels=w_values, yticklabels=k_values, ax=ax)
ax.set_ylabel('Number of clusters')
ax.set_xlabel('Window size')
ax.set_title('BIC')
plt.savefig('bics.png')
plt.close('all')

# Citations

1. U.S. Census Bureau and U.S. Bureau of Economic Analysis, Trade Balance: Goods, Balance of Payments Basis [BOPGTB], retrieved from FRED, Federal Reserve Bank of St. Louis; https://fred.stlouisfed.org/series/BOPGTB, May 11, 2026.
2. U.S. Bureau of Labor Statistics, Consumer Price Index for All Urban Consumers: All Items Less Food and Energy in U.S. City Average [CPILFESL], retrieved from FRED, Federal Reserve Bank of St. Louis; https://fred.stlouisfed.org/series/CPILFESL, May 11, 2026.
3. Board of Governors of the Federal Reserve System (US), Industrial Production: Total Index [INDPRO], retrieved from FRED, Federal Reserve Bank of St. Louis; https://fred.stlouisfed.org/series/INDPRO, May 11, 2026.
4. Board of Governors of the Federal Reserve System (US), M2 [M2SL], retrieved from FRED, Federal Reserve Bank of St. Louis; https://fred.stlouisfed.org/series/M2SL, May 11, 2026.
5. U.S. Bureau of Labor Statistics, Producer Price Index by Commodity: All Commodities [PPIACO], retrieved from FRED, Federal Reserve Bank of St. Louis; https://fred.stlouisfed.org/series/PPIACO, May 11, 2026.
6. U.S. Bureau of Labor Statistics, Unemployment Rate [UNRATE], retrieved from FRED, Federal Reserve Bank of St. Louis; https://fred.stlouisfed.org/series/UNRATE, May 11, 2026.
7. Federal Reserve Bank of Cleveland, 10-Year Real Interest Rate [REAINTRATREARAT10Y], retrieved from FRED, Federal Reserve Bank of St. Louis; https://fred.stlouisfed.org/series/REAINTRATREARAT10Y, May 11, 2026.